## This notebook performs a single science visit for this run for a target HD 164461
### This should be run AFTER performing focus on nearby star HD 185975
##### The script, slews and acquires the target, applies focus correction, takes sequence of data

#### WARNING: This currently requires ts_externalscripts branch **tickets/DM-29061**

In [2]:
TARGET="HD24661" #pole

In [1]:
import os
import sys
import asyncio
import time

import numpy as np
import logging 
import yaml
import matplotlib.pyplot as plt
import astropy

from lsst.ts import salobj
from lsst.ts.externalscripts.auxtel.latiss_cwfs_align import LatissCWFSAlign
from lsst.ts.externalscripts.auxtel.latiss_acquire_and_take_sequence import LatissAcquireAndTakeSequence

from lsst.ts.idl.enums.Script import ScriptState

In [3]:
print(os.environ["OSPL_URI"])
print(os.environ["LSST_DDS_PARTITION_PREFIX"])

file:///home/craiglagegit/WORK/ts_ddsconfig/config/ospl-shmem.xml
summit


In [4]:
# for autocompleted to work
%config IPCompleter.use_jedi = False

In [5]:
stream_handler = logging.StreamHandler(sys.stdout)
# if you want logging
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG

# turn off logging for matplotlib
mpl_logger = logging.getLogger('matplotlib')
mpl_logger.setLevel(logging.WARNING)

In [6]:
# make sure all remotes etc are running
script = LatissAcquireAndTakeSequence(index=1)  # this essentially calls the init method of the script
#await asyncio.sleep(10) # This can be removed in the future...
await script.start_task

DEBUG:Script.ATCS:atmcs: Adding all resources.
atmcs: Adding all resources.
DEBUG:Script.ATCS:atptg: Adding all resources.
atptg: Adding all resources.
DEBUG:Script.ATCS:ataos: Adding all resources.
ataos: Adding all resources.
DEBUG:Script.ATCS:atpneumatics: Adding all resources.
atpneumatics: Adding all resources.
DEBUG:Script.ATCS:athexapod: Adding all resources.
athexapod: Adding all resources.
DEBUG:Script.ATCS:atdome: Adding all resources.
atdome: Adding all resources.
DEBUG:Script.ATCS:atdometrajectory: Adding all resources.
atdometrajectory: Adding all resources.
DEBUG:Script.LATISS:atcamera: Adding all resources.
atcamera: Adding all resources.
DEBUG:Script.LATISS:atspectrograph: Adding all resources.
atspectrograph: Adding all resources.
DEBUG:Script.LATISS:atheaderservice: Adding all resources.
atheaderservice: Adding all resources.
DEBUG:Script.LATISS:atarchiver: Adding all resources.
atarchiver: Adding all resources.
INFO:ATPneumatics:Read historical data in 0.02 sec
Read 

In [7]:
# set wrap strategy
# this is required until the ATPtg is updated to not configure the mount for maximum time on target
script.atcs.rem.atptg.cmd_raDecTarget.set(azWrapStrategy=1)  # 1 does not unwrap, 0 unwraps

In [ ]:
# Do acquisition in the same mode we'll use to focus afterwards

In [9]:
acq1_yaml = yaml.safe_dump({"object_name": TARGET,
                            "do_acquire": True,
                            "acq_filter" : 'RG610',
                            "acq_grating" : 'empty_1', 
                            "acq_exposure_time": 0.5,
                            "max_acq_iter": 4,
                            "target_pointing_tolerance": 2,
                            "do_pointing_model": False,
                            "dataPath": '/project/shared/auxTel/rerun/quickLook',
                            "target_pointing_verification": False})
print(acq1_yaml)

acq_exposure_time: 0.5
acq_filter: RG610
acq_grating: empty_1
dataPath: /project/shared/auxTel/rerun/quickLook
do_acquire: true
do_pointing_model: false
max_acq_iter: 4
object_name: HD24661
target_pointing_tolerance: 2
target_pointing_verification: false



In [10]:
# Set script state to UNCONFIGURED
# this is required to run the script a 2nd time but otherwise is a no-op
script.set_state(ScriptState.UNCONFIGURED)
# Configure the script, which puts the ScriptState to CONFIGURED
acq1_configuration = script.cmd_configure.DataType()
acq1_configuration.config = acq1_yaml
await script.do_configure(acq1_configuration)

target DDS read queue is filling: 12 of 100 elements
target python read queue is filling: 32 of 100 elements


In [ ]:
# ATAOS must be on and corrections enabled, do as follows if required
# await script.atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=True)

In [11]:
# get group_id_data that will be used for all images for this target
time_for_groupID=astropy.time.Time.now().isot
group_id_data = script.cmd_setGroupId.DataType(
                groupId=time_for_groupID
            )
await script.do_setGroupId(group_id_data)

In [12]:
# This will align the target 
acq_results = await script.arun()

DEBUG:Script:Beginning target acquisition
Beginning target acquisition
DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): simbad.u-strasbg.fr:80
Starting new HTTP connection (1): simbad.u-strasbg.fr:80
DEBUG:urllib3.connectionpool:http://simbad.u-strasbg.fr:80 "POST /simbad/sim-script HTTP/1.1" 200 None
http://simbad.u-strasbg.fr:80 "POST /simbad/sim-script HTTP/1.1" 200 None
INFO:Script.ATCS:Slewing to HD24661: 03 54 17.3353 -27 40 07.849
Slewing to HD24661: 03 54 17.3353 -27 40 07.849
DEBUG:Script.ATCS:Setting rotator physical position to -105.0 deg. Rotator will track sky.
Setting rotator physical position to -105.0 deg. Rotator will track sky.
DEBUG:Script.ATCS:Parallactic angle: 106.20218809131738 | Sky Angle: 135.80772601371007
Parallactic angle: 106.20218809131738 | Sky Angle: 135.80772601371007
DEBUG:Script.ATCS:Sending command
Sending command
DEBUG:Script.ATCS:Stop tracking.
Stop tracking.
target python read queue is filling: 43 of 100 elements
DEBUG:Script.ATCS:Tr

### Now perform science data with 90lpmm Ronchi

In [13]:
seq1_yaml = yaml.safe_dump({"object_name": TARGET,
                                "do_acquire": False,
                                "do_take_sequence": True,
                                "exposure_time_sequence" : [10, 10, 10,
                                                            10, 10, 10,
                                                            10, 10, 10], 
                                "filter_sequence": ['RG610','RG610', 'RG610',
                                                    'BG40','BG40', 'BG40',
                                                    'empty_1','empty_1', 'empty_1'], 
                                # RG610 and Ronchi
                                "grating_sequence": ['ronchi90lpmm', 'ronchi90lpmm','ronchi90lpmm',
                                                     'ronchi90lpmm', 'ronchi90lpmm', 'ronchi90lpmm',
                                                     'ronchi90lpmm', 'ronchi90lpmm', 'ronchi90lpmm'],
                                "do_pointing_model": False,
                                "dataPath": '/project/shared/auxTel/rerun/quickLook',
                                "target_pointing_verification": False})
print(seq1_yaml)

dataPath: /project/shared/auxTel/rerun/quickLook
do_acquire: false
do_pointing_model: false
do_take_sequence: true
exposure_time_sequence:
- 10
- 10
- 10
- 10
- 10
- 10
- 10
- 10
- 10
filter_sequence:
- RG610
- RG610
- RG610
- BG40
- BG40
- BG40
- empty_1
- empty_1
- empty_1
grating_sequence:
- ronchi90lpmm
- ronchi90lpmm
- ronchi90lpmm
- ronchi90lpmm
- ronchi90lpmm
- ronchi90lpmm
- ronchi90lpmm
- ronchi90lpmm
- ronchi90lpmm
object_name: HD24661
target_pointing_verification: false



In [14]:
# Set script state to UNCONFIGURED
# this is required to run the script a 2nd time but otherwise is a no-op
script.set_state(ScriptState.UNCONFIGURED)
# Configure the script, which puts the ScriptState to CONFIGURED
seq1_configuration = script.cmd_configure.DataType()
seq1_configuration.config = seq1_yaml
await script.do_configure(seq1_configuration)

target python read queue is filling: 29 of 100 elements


In [15]:
# This take the sequence of images with the Ronchi
seq1_results = await script.arun()

DEBUG:Script:Beginning taking data for target sequence
Beginning taking data for target sequence
DEBUG:Script:Must load new filter RG610 or grating ronchi90lpmm
Must load new filter RG610 or grating ronchi90lpmm
DEBUG:Script:Instrument setup in take_sequence now waiting for 2*len[<ddsutil.ATSpectrograph_ackcmd_6d105732 object at 0x7f992c2b73d0>, <ddsutil.ATSpectrograph_ackcmd_6d105732 object at 0x7f992c2b70d0>] ATAOS events saying correction started/finished
Instrument setup in take_sequence now waiting for 2*len[<ddsutil.ATSpectrograph_ackcmd_6d105732 object at 0x7f992c2b73d0>, <ddsutil.ATSpectrograph_ackcmd_6d105732 object at 0x7f992c2b70d0>] ATAOS events saying correction started/finished
DEBUG:Script:Event #0 - evt_atspectrographCorrectionStarted is: private_revCode: e8ac7ff3, private_sndStamp: 1616543959.3212616, private_rcvStamp: 1616543959.321945, private_seqNum: 152, private_identity: ATAOS, private_origin: 51167, private_host: 0, focusOffset: -0.04200000315904617, pointingOffs

### Now re-acquire but using the hologram grating
##### This is the same position as the ronchi170lpmm grating

In [16]:
acq2_yaml = yaml.safe_dump({"object_name": TARGET,
                            "do_acquire": True,
                            "acq_filter" : 'RG610',
                            "acq_grating" : 'holo4_003', 
                            "acq_exposure_time": 0.5,
                            "max_acq_iter": 4,
                            "target_pointing_tolerance": 2,
                            "do_pointing_model": False,
                            "dataPath": '/project/shared/auxTel/rerun/quickLook',
                            "target_pointing_verification": False})
print(acq2_yaml)

acq_exposure_time: 0.5
acq_filter: RG610
acq_grating: holo4_003
dataPath: /project/shared/auxTel/rerun/quickLook
do_acquire: true
do_pointing_model: false
max_acq_iter: 4
object_name: HD24661
target_pointing_tolerance: 2
target_pointing_verification: false



In [17]:
# Set script state to UNCONFIGURED
# this is required to run the script a 2nd time but otherwise is a no-op
script.set_state(ScriptState.UNCONFIGURED)
# Configure the script, which puts the ScriptState to CONFIGURED
acq2_configuration = script.cmd_configure.DataType()
acq2_configuration.config = acq2_yaml
await script.do_configure(acq2_configuration)

target python read queue is filling: 25 of 100 elements


In [18]:
# This take the acquisition sequence for the hologram (and Ronchi170lpmm grating)
acq2_results = await script.arun()

DEBUG:Script:Beginning target acquisition
Beginning target acquisition
INFO:Script.ATCS:Slewing to HD24661: 03 54 17.3353 -27 40 07.849
Slewing to HD24661: 03 54 17.3353 -27 40 07.849
DEBUG:Script.ATCS:Setting rotator physical position to -105.0 deg. Rotator will track sky.
Setting rotator physical position to -105.0 deg. Rotator will track sky.
DEBUG:Script.ATCS:Parallactic angle: 106.46105839824746 | Sky Angle: 137.07753098592013
Parallactic angle: 106.46105839824746 | Sky Angle: 137.07753098592013
DEBUG:Script.ATCS:Sending command
Sending command
DEBUG:Script.ATCS:Stop tracking.
Stop tracking.
DEBUG:Script.ATCS:Tracking state: <AtMountState.TRACKINGENABLED: 9>
Tracking state: <AtMountState.TRACKINGENABLED: 9>
DEBUG:Script.ATCS:Tracking state: <AtMountState.STOPPING: 10>
Tracking state: <AtMountState.STOPPING: 10>
DEBUG:Script.ATCS:In Position: True.
In Position: True.
DEBUG:Script.ATCS:Scheduling check coroutines
Scheduling check coroutines
DEBUG:Script.ATCS:process as completed...


In [20]:
await script.atcs.reset_offsets()

DEBUG:Script.ATCS:Reseting absorbed offsets.
Reseting absorbed offsets.
DEBUG:Script.ATCS:Reseting non-absorbed offsets.
Reseting non-absorbed offsets.


### Now perform science data with 90lpmm Ronchi

In [ ]:
seq2_yaml = yaml.safe_dump({"object_name": TARGET,
                                "do_acquire": False,
                                "do_take_sequence": True,
                                "exposure_time_sequence" : [10, 10, 10,
                                                            10, 10, 10,
                                                            10, 10, 10], 
                                "filter_sequence": ['RG610','RG610', 'RG610',
                                                    'BG40','BG40', 'BG40',
                                                    'empty_1','empty_1', 'empty_1'], 
                                # RG610 and Ronchi
                                "grating_sequence": ['ronchi90lpmm', 'ronchi90lpmm','ronchi90lpmm',
                                                     'ronchi90lpmm', 'ronchi90lpmm', 'ronchi90lpmm',
                                                     'ronchi90lpmm', 'ronchi90lpmm', 'ronchi90lpmm'],
                                "do_pointing_model": False,
                                "dataPath": '/project/shared/auxTel/rerun/quickLook',
                                "target_pointing_verification": False})
print(seq2_yaml)

In [ ]:
# Set script state to UNCONFIGURED
# this is required to run the script a 2nd time but otherwise is a no-op
script.set_state(ScriptState.UNCONFIGURED)
# Configure the script, which puts the ScriptState to CONFIGURED
seq2_configuration = script.cmd_configure.DataType()
seq2_configuration.config = seq2_yaml
await script.do_configure(seq2_configuration)

In [ ]:
# This take the sequence of images with the Ronchi
seq2_results = await script.arun()